# BR04A — Generate reports with a leakage gate

Use this **in place of BR04** after BR03A passes. Run in each generator's working virtual environment. This uses the original loader, prompts, output paths and resumable journal. No bank rebuilding or BR01B restart is required.

A fresh target-free audit runs before generation. Every case is checked against its audited hash immediately before prompt formatting/inference or resume validation. An input/packet change aborts the call; it is not converted into an ordinary model failure. The gate writes its own protocol and trace next to `reports.jsonl`. Existing reports can resume, with retrospective checks explicitly distinguished from newly guarded inference.

A five-report pilot covers one query across the five arms. Inspect it, then set `MAX_NEW_REPORTS=0`. Successful reports are skipped; `RETRY_ERRORS=True` retries failed reports with unchanged settings. Signals, STOP files, allocation time and durable journal recovery retain their existing behavior.

In [ ]:
from pathlib import Path
import sys
EXP = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / 'config.example.json').is_file() and (p / 'common.py').is_file()), None)
if EXP is None: raise FileNotFoundError('Start Jupyter inside graph_reasoning/experiment02')
sys.path.insert(0, str(EXP))
from common import load_config, parent_setup, read
CONFIG_PATH = EXP / 'config.json'
if not CONFIG_PATH.is_file():
    raise FileNotFoundError('Run BR00 first to create experiment02/config.json')
CFG = load_config(CONFIG_PATH)
PROJECT = parent_setup(CFG)
RUN = Path(CFG['run_dir'])
print('Kernel:', sys.executable)
print('Cohort:', CFG['cohort'], '| outputs:', RUN)


In [ ]:
# Optional readiness check before loading a large model; the generation cell repeats it.
from extensions.leakage_controls import audit_packets
summary = audit_packets(CFG, max_run_hours=1.0)
print(summary)
if not summary['passed']:
    raise RuntimeError('Review BR03A leakage audit before loading the generator.')

In [ ]:
from generation import model_snapshot, load_generator
MODEL_KEY = 'pulse'  # pulse, llama_ecg, llama_base, qwen38
MODEL_PATH = None
DOWNLOAD_MODEL = False  # Optional authenticated Hugging Face download into existing cache.
TOKEN_FILE = '/path/to/private/hf_token'  # First line only; never printed.
ENABLE_THINKING = False  # Only Qwen has this switch; primary experiment is False.
LOAD_IN_4BIT = False
PULSE_REPO = None  # Your working PULSE repository when MODEL_KEY='pulse'.
MODEL_PATH = model_snapshot(MODEL_KEY, MODEL_PATH, download=DOWNLOAD_MODEL,
                            token_file=TOKEN_FILE, pin_dir=RUN / 'model_downloads')
runtime = load_generator(CFG, MODEL_KEY, MODEL_PATH, enable_thinking=ENABLE_THINKING,
                         load_in_4bit=LOAD_IN_4BIT, pulse_repo=PULSE_REPO)
print('Loaded:', runtime['model_id'], runtime['revision'])


In [ ]:
from generation import folder_for
from extensions.leakage_controls import guarded_generate
MAX_NEW_REPORTS = 5  # Inspect pilot, then 0 for the rest.
RETRY_ERRORS = False
MAX_RUN_HOURS = 30.0  # Audit + generation; less than allocation time remaining AFTER loading.
AUDIT_MAX_RUN_HOURS = 1.0
MAX_NEW_TOKENS = 1024 if MODEL_KEY == 'qwen38' else 384
print(guarded_generate(CFG, runtime, max_new=MAX_NEW_REPORTS, retry_errors=RETRY_ERRORS,
    max_run_hours=MAX_RUN_HOURS, max_new_tokens=MAX_NEW_TOKENS,
    audit_max_run_hours=AUDIT_MAX_RUN_HOURS))
FOLDER = folder_for(CFG, MODEL_KEY, ENABLE_THINKING, MAX_NEW_TOKENS)
print('Reports:', FOLDER / 'reports.jsonl')
print('Leakage trace:', FOLDER / 'leakage_guard_checks.jsonl')

In [ ]:
from common import latest
from collections import Counter
saved = latest(FOLDER / 'reports.jsonl')
failed = [r for r in saved.values() if r['status'] != 'ok']
print('Failures:', Counter(r.get('error') for r in failed))
for r in list(saved.values())[:5]: print(r['query_id'], r['arm'], r.get('text', r.get('error')))


When all selected generators finish, continue to **BR05** using the existing generation folders. The gate does not change report quality metrics or manufacture a label/clinical relevance judgment. Keep the audit summary and per-report trace with manuscript provenance.

Do not run BR04 and BR04A simultaneously for the same generator folder. Do not run any bank, retrieval or packet writer while guarded generation is active. Changed gate code or evidence requires investigation and a separately versioned run; immutable protocols are not silently rewritten.